# 00 — Setup & Data

Goals of this notebook:
1. Verify your environment (API keys, Neo4j, Qdrant)
2. Download the starter corpus (Wikipedia articles on AI labs and LLMs) into `data/raw/`
3. Review the golden Q&A set used by every other notebook

Run this once before notebooks 01–05. It's idempotent — safe to re-run.

## Why this corpus?

AI labs and large language models is a domain where each pattern has a clear shot:
- **Rich entity relationships** (Anthropic founded by ex-OpenAI staff, Transformer authors, model lineage) — KG-RAG territory
- **Exact terms** (model versions, paper titles, dates) — Hybrid territory
- **Multi-step questions** ("which lab was founded by people who left X and now builds Y") — Agentic territory

Swap in your own corpus by replacing `data/raw/` and editing `src/rag_compare/eval/golden.py`.

## 1. Environment check

In [ ]:
from rag_compare.common import get_settings

cfg = get_settings()
assert cfg.openai_api_key or cfg.anthropic_api_key, "Set an LLM API key in .env"
print("LLM model:", cfg.llm_model)
print("Embedding model:", cfg.embedding_model)
print("Neo4j:", cfg.neo4j_uri)
print("Qdrant:", cfg.qdrant_url)

In [ ]:
# Verify Neo4j is reachable (only needed for KG-RAG / notebook 01)
from neo4j import GraphDatabase

with GraphDatabase.driver(cfg.neo4j_uri, auth=(cfg.neo4j_username, cfg.neo4j_password)) as drv:
    drv.verify_connectivity()
print("Neo4j OK")

In [ ]:
# Verify Qdrant is reachable (needed for Hybrid / notebook 02)
from qdrant_client import QdrantClient

QdrantClient(url=cfg.qdrant_url, api_key=cfg.qdrant_api_key).get_collections()
print("Qdrant OK")

## 2. Download the starter corpus

Pulls ~18 Wikipedia articles via the public API. No API key needed. Idempotent — already-downloaded files are skipped.

In [ ]:
from rag_compare.common import STARTER_ARTICLES, fetch_starter_corpus

print(f"Will fetch {len(STARTER_ARTICLES)} articles:\n  - " + "\n  - ".join(STARTER_ARTICLES))
paths = fetch_starter_corpus()
print(f"\nCorpus on disk: {len(paths)} files")

In [ ]:
# Sanity-check the load path used by notebooks 01-05
from rag_compare.common import load_corpus

docs = load_corpus()
print(f"Loaded {len(docs)} documents")
print("First 200 chars of first doc:\n")
print(docs[0].text[:200])

## 3. Review the golden Q&A set

Each question targets a different *capability*. Notebook 04 runs every pipeline against this set so you can read off where each pattern wins.

Edit `src/rag_compare/eval/golden.py` to add your own.

In [ ]:
from rag_compare.eval import GOLDEN_QUESTIONS

for q in GOLDEN_QUESTIONS:
    print(f"[{q.capability:14}] {q.question}")
    print(f"               expected: {q.expected}\n")